In [2]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("practice_prompt1")

LangSmith 추적을 시작합니다.
[프로젝트명]
practice_prompt1


In [3]:
# 캐싱
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache

set_llm_cache(InMemoryCache())

In [4]:
# 간단한 template을 만들고 변수들 지정해보기
# PromptTemplate을 이용하되 .from_template 이용x
from langchain_core.prompts import PromptTemplate
from datetime import datetime
template = "오늘의 날짜는: {today} 입니다.오늘의 {place} 날씨를 알려주세요."

prompt = PromptTemplate(
    template=template,
    input_variables=['place'],
    partial_variables={'today': datetime.now().strftime("%Y-%m-%d")}
)

tmp = prompt.format(place='대전')


print(tmp)

오늘의 날짜는: 2026-03-13 입니다.오늘의 대전 날씨를 알려주세요.


In [22]:
# 위와같은 코드인데 좀 더 편리한
# from_template 사용
def get_today():
    return datetime.now().strftime("%Y-%m-%d")
prompt = PromptTemplate.from_template(template)

prompt = prompt.partial(today=get_today())

tmp = prompt.format(place='대전')

print(tmp)

오늘의 날짜는: 2026-03-13 입니다.오늘의 대전 날씨를 알려주세요.


In [ ]:
# 파일에서 불러오기
from langchain_core.prompts import load_prompt

path = 'data/test_prompt1.yaml'
prompt = load_prompt(path, encoding='utf-8')
prompt = prompt.partial(today=get_today)
tmp = prompt.format(place='대전')
tmp



'오늘의 날짜는: 2026-03-13 입니다. 대전의 날씨를 알려주세요.'

In [24]:
# Chatting 형식으로 만들기
from langchain_core.prompts import ChatPromptTemplate

dialog = [
        ('system', """당신은 응답에 능한 도우미 입니다. 당신은 {country}에 살고 있고
         마지막 question에 답하시오."""),
        ('human', '안녕?'),
        ('ai', '반가워요!, 무엇을 도와드릴까요?'),
        ('human', '{question}')
    ]

prompt = ChatPromptTemplate.from_messages(dialog)
messages = prompt.format_messages(
    country= '대전',
    question= '당신은 어디에 살고 있습니까?'
)

messages

[SystemMessage(content='당신은 응답에 능한 도우미 입니다. 당신은 대전에 살고 있고\n         마지막 question에 답하시오.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='안녕?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='반가워요!, 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='당신은 어디에 살고 있습니까?', additional_kwargs={}, response_metadata={})]

In [25]:
# 위의 dialog를 message holder 사용
from langchain_core.prompts import MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 응답에 능한 도우미 입니다. 당신은 대전에 삽니다'),
    MessagesPlaceholder(variable_name='conversation'),
    ('human', '{question}')
])

filled_prompt = prompt.format_messages(
    conversation=[
        ('human', '안녕?'),
        ('ai', '안녕하세요! 무엇을 도와드릴까요?')
    ],
    question='당신의 이름은 무엇입니까?'
)

filled_prompt

[SystemMessage(content='당신은 응답에 능한 도우미 입니다. 당신은 대전에 삽니다', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='안녕?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='당신의 이름은 무엇입니까?', additional_kwargs={}, response_metadata={})]

In [27]:
from langchain_core.prompts import FewShotPromptTemplate

examples = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
""",
    },
    {
        "question": "네이버의 창립자는 언제 태어났나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일
""",
    },
    {
        "question": "율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 율곡 이이의 어머니는 누구인가요?
중간 답변: 율곡 이이의 어머니는 신사임당입니다.
추가 질문: 신사임당은 언제 태어났나요?
중간 답변: 신사임당은 1504년에 태어났습니다.
추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
최종 답변은: 연산군
""",
    },
    {
        "question": "올드보이와 기생충의 감독이 같은 나라 출신인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 올드보이의 감독은 누구인가요?
중간 답변: 올드보이의 감독은 박찬욱입니다.
추가 질문: 박찬욱은 어느 나라 출신인가요?
중간 답변: 박찬욱은 대한민국 출신입니다.
추가 질문: 기생충의 감독은 누구인가요?
중간 답변: 기생충의 감독은 봉준호입니다.
추가 질문: 봉준호는 어느 나라 출신인가요?
중간 답변: 봉준호는 대한민국 출신입니다.
최종 답변은: 예
""",
    },
]
example_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix='#Question:\n{question}\nAnswer:',
    input_variables=['question']
)

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
print(prompt.format(question=question))

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인


Question:
네이버의 창립자는 언제 태어났나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일


Question:
율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 율곡 이이의 어머니는 누구인가요?
중간 답변: 율곡 이이의 어머니는 신사임당입니다.
추가 질문: 신사임당은 언제 태어났나요?
중간 답변: 신사임당은 1504년에 태어났습니다.
추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
최종 답변은: 연산군


Question:
올드보이와 기생충의 감독이 같은 나라 출신인가요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 올드보이의 감독은 누구인가요?
중간 답변: 올드보이의 감독은 박찬욱입니다.
추가 질문: 박찬욱은 어느 나라 출신인가요?
중간 답변: 박찬욱은 대한민국 출신입니다.
추가 질문: 기생충의 감독은 누구인가요?
중간 답변: 기생충의 감독은 봉준호입니다.
추가 질문: 봉준호는 어느 나라 출신인가요?
중간 답변: 봉준호는 대한민국 출신입니다.
최종 답변은: 예


#Question:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:


In [28]:
# example selector 사용
from langchain_core.example_selectors import(
    MaxMarginalRelevanceExampleSelector,
    SemanticSimilarityExampleSelector
)

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

example_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)
selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    OpenAIEmbeddings(),
    Chroma(),
    k=1,
)

selected_examples = selector.select_examples(
    {'question': question}
)


print(question)
print(selected_examples[0]['answer'])

Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일



In [32]:
# FewShotChatMessagePromptTemplate
from langchain_core.prompts import FewShotChatMessagePromptTemplate
examples = [
    {
        "instruction": "당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요",
        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.",
        "answer": """
회의록: XYZ 회사 마케팅 전략 회의
일시: 2023년 12월 25일
장소: XYZ 회사 회의실
참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
   - 회의는 김수진 팀장의 개회사로 시작됨.
   - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.

2. 시장 동향 개요 (김수진)
   - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.
   - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.

3. 디지털 마케팅 전략 (박지민)
   - 박지민은 디지털 마케팅 전략에 대해 발표.
   - 온라인 광고와 SEO 최적화 방안에 중점을 둠.

4. 소셜 미디어 캠페인 (이준호)
   - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.
   - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.

5. 종합 논의
   - 팀원들 간의 아이디어 공유 및 토론.
   - 각 전략에 대한 예산 및 자원 배분에 대해 논의.

6. 마무리
   - 다음 회의 날짜 및 시간 확정.
   - 회의록 정리 및 배포는 박지민 담당.
""",
    },
    {
        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. 보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 도시 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 도시 개발 사례를 여러 국가에서 소개하고, 이러한 사례들을 통해 얻은 교훈을 요약하고 있습니다.",
        "answer": """
문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서

- 중요성: 지속 가능한 도시 개발이 필수적인 이유와 그에 따른 사회적, 경제적, 환경적 이익을 강조.
- 현 문제점: 현재의 도시화 과정에서 발생하는 주요 문제점들, 예를 들어 환경 오염, 자원 고갈, 불평등 증가 등을 분석.
- 전략: 지속 가능한 도시 개발을 달성하기 위한 다양한 전략 제시. 이에는 친환경 건축, 대중교통 개선, 에너지 효율성 증대, 지역사회 참여 강화 등이 포함됨.
- 사례 연구: 전 세계 여러 도시의 성공적인 지속 가능한 개발 사례를 소개. 예를 들어, 덴마크의 코펜하겐, 일본의 요코하마 등의 사례를 통해 실현 가능한 전략들을 설명.
- 교훈: 이러한 사례들에서 얻은 주요 교훈을 요약. 강조된 교훈에는 다각적 접근의 중요성, 지역사회와의 협력, 장기적 계획의 필요성 등이 포함됨.

이 보고서는 지속 가능한 도시 개발이 어떻게 현실적이고 효과적인 형태로 이루어질 수 있는지에 대한 심도 있는 분석을 제공합니다.
""",
    },
    {
        "instruction": "당신은 문장 교정 전문가 입니다. 다음 주어진 문장을 교정해 주세요",
        "input": "우리 회사는 새로운 마케팅 전략을 도입하려고 한다. 이를 통해 고객과의 소통이 더 효과적이 될 것이다.",
        "answer": "본 회사는 새로운 마케팅 전략을 도입함으로써, 고객과의 소통을 보다 효과적으로 개선할 수 있을 것으로 기대된다.",
    },
]
chroma = Chroma('fewshot_chat', OpenAIEmbeddings())

example_prompt = ChatPromptTemplate.from_messages(
    [
        ('human', '{instruction}:\n{input}'),
        ('ai', '{answer}')
    ]
)

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    OpenAIEmbeddings(),
    chroma,
    k=1.
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
)

final_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 훌륭한 도우미입니다.'),
    few_shot_prompt,
    ('human', '{instruction}\n{input}')
])

In [ ]:
"""
customexampleselector는 지시사항이 많거나 애매할때..? 사용
좀더 사용하기 까다롭고 귀찮음.
대부분의 경우 위에서 쓴 semantic어쩌구 그거 씀.
"""


ChatPromptTemplate(input_variables=['input', 'instruction'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 훌륭한 도우미입니다.'), additional_kwargs={}), FewShotChatMessagePromptTemplate(example_selector=SemanticSimilarityExampleSelector(vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001468E1129F0>, k=1, example_keys=None, input_keys=None, vectorstore_kwargs=None), input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['answer', 'input', 'instruction'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input', 'instruction'], input_types={}, partial_variables={}, template='{instruction}:\n{input}'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['answer'], input_types={}, partial_variables={}, templa

In [ ]:
# hub에서 프롬프트 가져오기
# rlm/rag-prompt
from langchain_classic import hub

prompt = hub.pull("rlm/rag-prompt")
print(prompt)

In [18]:
# 특정 버전을 가져오기
# rlm/rag-prompt:50442af1
prompt = hub.pull("rlm/rag-prompt:50442af1")
prompt


ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [34]:
# prompt hub에 자신의 프롬프트 등록
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "주어진 내용을 바탕으로 다음 문장을 요약하세요. 답변은 반드시 한글로 작성하세요\n\nCONTEXT:{context}\n\nSUMMARY:"
)
hub.push("simple-summary-korean2", prompt)

'https://smith.langchain.com/prompts/simple-summary-korean2/0417a037?organizationId=faf98a37-3f9b-4afa-a5a4-6a25ffc8fde9'

In [28]:
# 내가 올린 prompt가져오기
pulled_prompt = hub.pull("simple-summary-korean")

In [29]:
print(pulled_prompt)

input_variables=['context'] input_types={} partial_variables={} metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'simple-summary-korean', 'lc_hub_commit_hash': '0417a037bf6179248d847385d392d4a9ff909084e1735f6e04528cf19f4ba0e2'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='주어진 내용을 바탕으로 다음 문장을 요약하세요. 답변은 반드시 한글로 작성하세요\n\nCONTEXT:{context}\n\nSUMMARY:'), additional_kwargs={})]
